# Baseline notebook: customer churn framing
Purpose: show what a **non-ML baseline** achieves on the 12-row sample *before* any model is justified. Metrics here are descriptive, not evidence of generalisation.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, brier_score_loss
d = pd.read_csv('churn.csv'); y = d.churned
print(d.shape, 'missing:', int(d.isna().sum().sum()), 'dup rows:', int(d.duplicated().sum()), 'dup ids:', int(d.customer_id.duplicated().sum()))
print('churn rate:', round(y.mean(),3), f'({y.sum()} of {len(d)})')

(12, 7) missing: 0 dup rows: 0 dup ids: 0
churn rate: 0.417 (5 of 12)


## Data quality and separability

In [2]:
print(d.groupby('plan_type').churned.agg(['count','sum']))
print(d.groupby('support_tickets').churned.agg(['count','sum']))
print(d.groupby('churned')[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']].agg(['min','max']).T)

           count  sum
plan_type            
Basic          5    5
Pro            3    0
Standard       4    0
                 count  sum
support_tickets            
0                    2    0
1                    2    0
2                    3    0
3                    2    2
4                    2    2
5                    1    1
churned                   0    1
tenure_months     min     7    1
                  max    30    8
support_tickets   min     0    3
                  max     2    5
monthly_spend_inr min   799  499
                  max  1499  799
last_login_days   min     1   12
                  max     9   30


Churners and non-churners do not overlap on tickets (>=3), last login (>=12 vs <=9 days) or plan (all Basic). Perfect separation on 12 rows is a warning about sample size and possible leakage, not proof of a strong model.

In [3]:
def rep(name, p):
    tn, fp, fn, tp = confusion_matrix(y, p).ravel()
    print(f'{name:34s} TP={tp} FP={fp} FN={fn} TN={tn}')
rep('Majority class (predict no churn)', np.zeros(len(d), int))
rep('Rule: support_tickets >= 3', (d.support_tickets >= 3).astype(int))
rep('Rule: last_login_days >= 10', (d.last_login_days >= 10).astype(int))
rep('Rule: plan_type == Basic', (d.plan_type == 'Basic').astype(int))
rep('Rule: tickets>=3 OR login>=10', ((d.support_tickets >= 3) | (d.last_login_days >= 10)).astype(int))

Majority class (predict no churn)  TP=0 FP=0 FN=5 TN=7
Rule: support_tickets >= 3         TP=5 FP=0 FN=0 TN=7
Rule: last_login_days >= 10        TP=5 FP=0 FN=0 TN=7
Rule: plan_type == Basic           TP=5 FP=0 FN=0 TN=7
Rule: tickets>=3 OR login>=10      TP=5 FP=0 FN=0 TN=7


## ML comparison: leave-one-out logistic regression
With 12 rows, LOO is the only defensible split, and it is still extremely noisy.

In [4]:
X = d[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']]
m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0))
p = cross_val_predict(m, X, y, cv=LeaveOneOut(), method='predict_proba')[:, 1]
rep('LOO logistic regression (thr 0.5)', (p >= .5).astype(int))
print('Brier model:', round(brier_score_loss(y, p),3), '| Brier constant-rate:', round(brier_score_loss(y, np.full(len(d), y.mean())),3))
print('LOO probabilities:', np.round(p, 2))

LOO logistic regression (thr 0.5)  TP=5 FP=1 FN=0 TN=6
Brier model: 0.059 | Brier constant-rate: 0.243
LOO probabilities: [0.94 0.04 0.58 0.   0.25 0.99 0.06 0.74 0.11 0.7  0.04 0.54]


## Reading the result
- A one-line rule matches the model's recall with zero false positives on this sample; the model adds one false positive.
- ML is **not justified yet**: use the rule (or a hand-reviewed watchlist) until a larger, time-split dataset exists.
- Before any retraining: check that `support_tickets` and `last_login_days` are measured **before** the prediction date (see data card, leakage).